# Chronos-2 -> ONNX (working export)

Run top to bottom. No paths to fill in, no timestamps.

Three differences from the first export:
1. `group_ids = zeros` so all rows form **one** task: row 0 is the target, rows 1.. are covariates. Without it the model treats every row as an unrelated series.
2. `num_output_patches = 2` so the horizon is 22, not 16.
3. `dynamic_shapes` instead of `dynamic_axes`, and traced with 4 rows instead of 1, so the row count is not welded shut.

The last cell prints exactly what the C++ side has to feed. This exports the **base** small model (zero-shot). To use fine-tuned weights later, change `MODEL` to the folder they were saved in - nothing else in the notebook changes.

In [ ]:
%pip install -q chronos-forecasting onnx onnxruntime onnxscript

In [ ]:
import math
import numpy as np
import onnx
import onnxruntime as ort
import torch
from chronos import Chronos2Pipeline

MODEL   = "autogluon/chronos-2-small"   # or a local folder with fine-tuned weights
OUT     = "chronos2.onnx"
HORIZON = 21        # study horizon
PATCH   = 16        # model_output_patch_size
NOP     = math.ceil(HORIZON / PATCH)    # 2 output patches -> 32 steps -> sliced to 22

pipe  = Chronos2Pipeline.from_pretrained(MODEL, device_map="cpu", dtype=torch.float32)
model = pipe.model.eval()

QGRID = list(pipe.quantiles)
qidx  = lambda level: min(range(len(QGRID)), key=lambda i: abs(QGRID[i] - level))

print("quantile grid:", QGRID)
print("C++ indices ->  0.1: %d   0.5 (median): %d   0.9: %d" % (qidx(0.1), qidx(0.5), qidx(0.9)))

In [ ]:
class Wrap(torch.nn.Module):
    """context [V, L] -> quantiles [V, Q, HORIZON].  row 0 = target, rows 1.. = covariates."""

    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, context):
        gids = torch.zeros(context.shape[0], dtype=torch.long, device=context.device)
        out = self.m(context, context_mask=None, group_ids=gids, num_output_patches=NOP)
        return out.quantile_preds[..., :HORIZON]


wrapped = Wrap(model).eval()

# the attention-mask path feeds bool tensors to einsum, which the exporter rejects
_orig_einsum = torch.einsum

def _einsum_f32(*a, **k):
    eq, ops = (a[0], a[1:]) if isinstance(a[0], str) else (None, a)
    ops = [o.to(torch.float32) if isinstance(o, torch.Tensor) and o.dtype == torch.bool else o
           for o in ops]
    return _orig_einsum(eq, *ops, **k) if eq is not None else _orig_einsum(*ops, **k)


def export(rows, dynamic_rows):
    """Export at `rows`. Returns (torch reference output, the input used)."""
    dummy  = torch.randn(rows, 512, dtype=torch.float32)
    shapes = {"context": {1: torch.export.Dim("ctx", min=64, max=8192)}}
    if dynamic_rows:
        shapes["context"][0] = torch.export.Dim("variates", min=2, max=64)

    torch.einsum = _einsum_f32
    try:
        with torch.no_grad():
            ref = wrapped(dummy).numpy()
        torch.onnx.export(
            wrapped, (dummy,), OUT,
            opset_version=18,
            input_names=["context"],
            output_names=["quantiles"],
            dynamic_shapes=shapes,
            dynamo=True,
        )
    finally:
        torch.einsum = _orig_einsum
    return ref, dummy.numpy()


def check(ref, ref_in, rows_to_try):
    """Proves the graph by running it, not by reading its shapes."""
    s   = ort.InferenceSession(OUT, providers=["CPUExecutionProvider"])
    err = float(np.abs(s.run(None, {"context": ref_in})[0] - ref).max())
    print("torch vs onnx max abs err: %.3e   (want < 1e-4)" % err)
    ok = err < 1e-4
    for v in rows_to_try:
        for L in (256, 512, 1000):
            try:
                o    = s.run(None, {"context": np.random.randn(v, L).astype(np.float32)})[0]
                good = o.shape[0] == v and o.shape[2] == HORIZON
                print("   V=%-3d L=%-5d -> %s%s" % (v, L, o.shape, "" if good else "   <-- WRONG"))
                ok &= good
            except Exception as e:
                print("   V=%-3d L=%-5d -> FAILED: %s" % (v, L, str(e).splitlines()[0][:80]))
                ok = False
    return ok

In [ ]:
FIXED_V = 17     # used only if a dynamic row count turns out to be impossible

print("=== exporting with a dynamic number of rows ===")
dynamic_ok = False
try:
    ref, ref_in = export(rows=4, dynamic_rows=True)
    dynamic_ok  = check(ref, ref_in, rows_to_try=[3, 7, 17])
except Exception as e:
    print("export failed:", str(e).splitlines()[0][:200])

if not dynamic_ok:
    print("\n=== falling back: rows frozen at %d ===" % FIXED_V)
    ref, ref_in = export(rows=FIXED_V, dynamic_rows=False)
    check(ref, ref_in, rows_to_try=[FIXED_V])

weights = {kv.value
           for i in onnx.load(OUT, load_external_data=False).graph.initializer
           for kv in i.external_data if kv.key == "location"}

print("\n" + "=" * 64)
print("ROWS           : %s" % ("any 2..64" if dynamic_ok else "exactly %d" % FIXED_V))
print("CONTEXT LENGTH : any 64..8192")
print("INPUT  context   [V, L]        row 0 = target, rows 1.. = covariates, same order every call")
print("OUTPUT quantiles [V, %d, %d]    read row 0, quantile index %d" % (len(QGRID), HORIZON, qidx(0.5)))
print("FILES          : keep %s next to %s" % (weights or "(no separate weights file)", OUT))
print("INPUT VALUES   : raw log-RV. do NOT normalise, the graph does it internally. NaN is allowed.")
print("=" * 64)